# Single Match Analysis - Final Match




## 0. Setup - Imports and Configuration

In [18]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from src.analysis.game_utils import game_summary
from src.analysis.prediction_utils import normalize_predictions, print_top_predictions, get_top_indices, create_top_sequences, visualize_top_sequences
from src.data.data_loader import load_socceraction_match
from src.ml.models.model_factory import load_model
from src.ml.preprocessing.sequence_preprocessor import SequencePreprocessor, PreprocessingMode
from src.ml.preprocessing.xthreat import get_default_xt_model


In [19]:
# Configure pandas display options
pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

## 1. Load ratings

In [20]:
transformer_ratings = pd.read_csv("generated/ratings/player_contributions_match_3943043_transformer.csv")
print(transformer_ratings.columns)
transformer_ratings.head()

Index(['total_value', 'num_sequences', 'avg_value', 'max_value', 'player_id', 'player_name', 'team_name'], dtype='object')


,total_value,num_sequences,avg_value,max_value,player_id,player_name,team_name
0,1.720473,102,0.016867,0.179595,68574,Nicholas Williams Arthuer,Spain
1,1.222608,71,0.017220,0.919108,3943,Declan Rice,England
2,0.817241,75,0.010897,0.187941,316046,Lamine Yamal Nasraoui Ebana,Spain
3,0.798686,125,0.006389,0.185811,6655,Fabián Ruiz Peña,Spain
4,0.682922,77,0.008869,0.180763,3205,Kyle Walker,England


In [21]:
attention_lstm_ratings = pd.read_csv("generated/ratings/player_contributions_match_3943043_attention_lstm.csv")
attention_lstm_ratings.head()

,total_value,num_sequences,avg_value,max_value,player_id,player_name,team_name
0,1.242785,102,0.012184,0.118721,68574,Nicholas Williams Arthuer,Spain
1,0.962968,125,0.007704,0.055847,6655,Fabián Ruiz Peña,Spain
2,0.776907,126,0.006166,0.050386,5721,Daniel Carvajal Ramos,Spain
3,0.709577,70,0.010137,0.035139,16532,Daniel Olmo Carvajal,Spain
4,0.660164,152,0.004343,0.058445,4353,Aymeric Laporte,Spain


In [22]:
sofascore_rating = pd.read_csv("generated/ratings/sofacore.csv")
sofascore_rating

,player_name,team_name,Pozycja,sofascore_rating
0,Nicholas Williams Arthuer,Spain,Pomocnik,7.9
1,Jude Bellingham,England,Pomocnik,7.7
2,Lamine Yamal Nasraoui Ebana,Spain,Pomocnik,7.5
3,Mikel Oyarzabal Ugarte,Spain,Napastnik,7.5
4,Fabián Ruiz Peña,Spain,Pomocnik,7.4
5,Jordan Pickford,England,Bramkarz,7.4
6,Cole Palmer,England,Napastnik,7.3
7,Marc Cucurella Saseta,Spain,Obrońca,7.1
8,Aymeric Laporte,Spain,Obrońca,7.0
9,Martín Zubimendi Ibáñez,Spain,Pomocnik,7.0


## 2. Analysis


In [23]:
# Merge all ratings
attention_lstm_prep = attention_lstm_ratings[['player_name', 'team_name', 'total_value']].rename(columns={
    'total_value': 'attention_lstm_total_value',
})

transformer_prep = transformer_ratings[['player_name', 'team_name', 'total_value', 'num_sequences']].rename(columns={
    'total_value': 'transformer_total_value',
})

sofascore_prep = sofascore_rating[['player_name', 'team_name', 'sofascore_rating']]

# Merge all dataframes
all_ratings = attention_lstm_prep.merge(
    transformer_prep,
    on=['player_name', 'team_name'],
    how='outer'
).merge(
    sofascore_prep,
    on=['player_name', 'team_name'],
    how='outer'
)

# Convert num_sequences columns to int
all_ratings['num_sequences'] = all_ratings['num_sequences'].astype(int)

# Sort by attention_lstm_total_value
all_ratings = all_ratings.sort_values('attention_lstm_total_value', ascending=False)

all_ratings

,player_name,team_name,attention_lstm_total_value,transformer_total_value,num_sequences,sofascore_rating
20,Nicholas Williams Arthuer,Spain,1.242785,1.720473,102,7.9
6,Fabián Ruiz Peña,Spain,0.962968,0.798686,125,7.4
3,Daniel Carvajal Ramos,Spain,0.776907,0.665989,126,6.7
4,Daniel Olmo Carvajal,Spain,0.709577,0.506555,70,6.9
0,Aymeric Laporte,Spain,0.660164,0.568710,152,7.0
23,Robin Aime Robert Le Normand,Spain,0.658067,0.431317,153,6.9
14,Lamine Yamal Nasraoui Ebana,Spain,0.571512,0.817241,75,7.5
5,Declan Rice,England,0.531927,1.222608,71,6.8
11,Jude Bellingham,England,0.528454,0.468874,66,7.7
1,Bukayo Saka,England,0.410017,0.448848,56,6.8


### Save to csv

In [24]:
all_ratings.to_csv("generated/ratings/all_ratings.csv")

In [25]:
all_ratings.columns


Index(['player_name', 'team_name', 'attention_lstm_total_value', 'transformer_total_value', 'num_sequences', 'sofascore_rating'], dtype='object')